# 국가법령정보센터 Open API로 법령 가져오기

법제처 Open API의 법령 목록 조회(`lawSearch.do`)로 검색어에 맞는 법령을 찾고, 선택한 법령의 일련번호(`MST`)로 본문 조회(`lawService.do`)를 호출합니다.

- 공식 매뉴얼: <https://open.law.go.kr/LSO/openApi/openApiManual.do>
- 법령 검색 API: `https://www.law.go.kr/DRF/lawSearch.do`
- 법령 본문 API: `https://www.law.go.kr/DRF/lawService.do`

## 사전 설정

1. Open API 사이트에서 사용 신청 후 발급/확인한 **OC 값**을 준비합니다. OC는 이 API에서 사용하는 사용자 식별값이며, OpenAI API 키와는 다릅니다.
2. 프로젝트 루트 `.env`에 다음 항목을 직접 추가합니다. 실제 OC 값은 코드나 채팅에 공개하지 마세요.

```text
LAW_API_OC=여기에_발급받은_OC_값
```

이 노트북은 XML 응답을 표준 라이브러리로 읽으므로 새 패키지가 필요하지 않습니다.

In [6]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from dotenv import load_dotenv

load_dotenv(override=True)
LAW_API_OC = os.getenv("LAW_API_OC", "").strip()

if not LAW_API_OC:
    raise ValueError(
        "LAW_API_OC가 설정되지 않았습니다. 프로젝트 루트 .env에 "
        "LAW_API_OC=발급받은_OC_값 을 추가한 뒤 커널을 재시작하세요."
    )

print("법령 Open API OC 설정 확인 완료")

법령 Open API OC 설정 확인 완료


## 1. 검색어 설정 및 API 요청 함수

현행법령 검색은 `target="eflaw"`를 사용합니다. `target="law"`로 호출하면 검색 결과가 0건이 될 수 있습니다. 필요에 따라 검색어, 페이지, 페이지당 결과 수를 바꿔 보세요.

In [7]:
API_BASE_URL = "https://www.law.go.kr/DRF"


def call_law_api(endpoint: str, params: dict[str, str]) -> ET.Element:
    """국가법령정보센터 API를 호출하고 XML 루트 요소를 반환합니다."""
    query_params = {"OC": LAW_API_OC, "type": "XML", **params}
    url = f"{API_BASE_URL}/{endpoint}.do?{urlencode(query_params)}"
    request = Request(url, headers={"Accept": "application/xml"})

    with urlopen(request, timeout=30) as response:
        response_body = response.read()

    root = ET.fromstring(response_body)
    result_code = root.findtext("resultCode")
    if result_code and result_code != "00":
        result_message = root.findtext("resultMsg", "응답 메시지가 없습니다.")
        raise RuntimeError(
            f"법령 API 오류: resultCode={result_code}, resultMsg={result_message}"
        )

    return root

In [8]:
QUERY = "인공지능"
PAGE = 1
DISPLAY = 20

search_xml = call_law_api(
    "lawSearch",
    {
        "target": "eflaw",
        "query": QUERY,
        "display": str(DISPLAY),
        "page": str(PAGE),
    },
)

total_count = int(search_xml.findtext("totalCnt", "0"))
law_results = []

for law in search_xml.findall("law"):
    law_results.append(
        {
            "mst": law.findtext("법령일련번호", ""),
            "name": law.findtext("법령명한글", ""),
            "abbreviation": law.findtext("법령약칭명", ""),
            "type": law.findtext("법령구분명", ""),
            "promulgation_date": law.findtext("공포일자", ""),
            "effective_date": law.findtext("시행일자", ""),
            "department": law.findtext("소관부처명", ""),
        }
    )

SEARCH_XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'law_search_result.xml'
SEARCH_XML_PATH.parent.mkdir(parents=True, exist_ok=True)
SEARCH_XML_PATH.write_bytes(ET.tostring(search_xml, encoding='utf-8', xml_declaration=True))
print(f'검색 결과 XML 저장 완료: {SEARCH_XML_PATH}')

print(f"검색어: {QUERY} / 전체 검색 결과: {total_count}건")
for index, law in enumerate(law_results):
    print(
        f"[{index}] {law['name']} ({law['type']}) | "
        f"시행일 {law['effective_date']} | MST={law['mst']}"
    )

검색어: 인공지능 / 전체 검색 결과: 23건
[0] 국가인공지능위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20250904 | MST=273613
[1] 국가인공지능위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20240806 | MST=264599
[2] 국가인공지능전략위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20260122 | MST=282881
[3] 국가인공지능전략위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20260102 | MST=280929
[4] 국가인공지능전략위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20251001 | MST=278203
[5] 국가인공지능전략위원회의 설치 및 운영에 관한 규정 (대통령령) | 시행일 20250904 | MST=273611
[6] 산업 디지털 전환 및 인공지능 활용 촉진법 (법률) | 시행일 20260918 | MST=284365
[7] 산업 디지털 전환 및 인공지능 활용 촉진법 (법률) | 시행일 20260701 | MST=281883
[8] 산업 디지털 전환 및 인공지능 활용 촉진법 시행규칙 (산업통상부령) | 시행일 20260701 | MST=288055
[9] 산업 디지털 전환 및 인공지능 활용 촉진법 시행령 (대통령령) | 시행일 20260701 | MST=287399
[10] 인공지능 데이터센터 산업 진흥에 관한 특별법 (법률) | 시행일 20270310 | MST=286707
[11] 인공지능 및 데이터 기반 행정 활성화에 관한 법률 (법률) | 시행일 20270228 | MST=283735
[12] 인공지능 및 데이터 기반 행정 활성화에 관한 법률 (법률) | 시행일 20260828 | MST=283735
[13] 인공지능 및 데이터 기반 행정 활성화에 관한 법률 시행규칙 (행정안전부령) | 시행일 20260828 | MST=289193
[14] 인공지능 및 데이터 기반 행정 활성화에 관한 법률 시행령 (대통령령) | 시

- [16] 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 (법률) | 시행일 20260721 | MST=282791

- SELECT_INDEX = 16  # 16 이라서

- https://www.law.go.kr/lsSc.do?section=&menuId=1&subMenuId=15&tabMenuId=81&eventGubun=060101&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5#undefined

## 2. 검색 결과에서 법령을 골라 본문 가져오기

위 검색 결과의 `[번호]` 중 가져올 법령의 번호를 `SELECT_INDEX`에 지정합니다. 기본값 `0`은 첫 번째 검색 결과입니다. 상세 API는 법령 목록 응답의 법령일련번호를 `ID` 파라미터로 받습니다.

In [11]:
if not law_results:
    raise ValueError("검색 결과가 없습니다. QUERY를 바꾸거나 PAGE를 확인하세요.")

SELECT_INDEX = 16  # 16 이라서
if not 0 <= SELECT_INDEX < len(law_results):
    raise IndexError(
        f"SELECT_INDEX={SELECT_INDEX}는 현재 검색 결과 범위를 벗어났습니다. "
        f"검색 결과 인덱스: 0부터 {len(law_results) - 1}"
    )

selected_law = law_results[SELECT_INDEX]
detail_xml = call_law_api(
    "lawService",
    {
        "target": "eflaw",
        "MST": selected_law["mst"],
        "efYd": selected_law["effective_date"],
    },
)

print(f"가져온 법령: {selected_law['name']}")
print(f"법령 구분: {selected_law['type']}")
print(f"공포일: {selected_law['promulgation_date']}")
print(f"시행일: {selected_law['effective_date']}")
print(f"상세 응답 최상위 태그: {detail_xml.tag}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_DIR = PROJECT_ROOT / 'data' / 'ai_basic_law'
XML_DIR.mkdir(parents=True, exist_ok=True)
XML_PATH = XML_DIR / 'ai_basic_law.xml'
XML_PATH.write_bytes(ET.tostring(detail_xml, encoding='utf-8', xml_declaration=True))
print(f'XML 저장 완료: {XML_PATH}')

가져온 법령: 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법
법령 구분: 법률
공포일: 20260120
시행일: 20260721
상세 응답 최상위 태그: Law


## 3. 본문을 조문 단위 데이터로 정리

`documents`는 조문 번호·제목·내용으로 구성한 리스트입니다. 이후 임베딩이나 Qdrant 저장 단계에서 문서 입력으로 활용할 수 있습니다. 조문 전체를 출력하면 길 수 있으므로 아래 셀은 개수와 첫 조문만 미리 보여줍니다.

In [12]:
CONTENT_TAGS = ("조문내용", "항내용", "호내용", "목내용")
documents = []

for unit in detail_xml.findall(".//조문단위"):
    number = unit.findtext("조문번호", "").strip()
    title = unit.findtext("조문제목", "").strip()
    content_parts = []

    for tag in CONTENT_TAGS:
        for element in unit.findall(f"./{tag}"):
            text = " ".join("".join(element.itertext()).split())
            if text and text not in content_parts:
                content_parts.append(text)

    content = "\n".join(content_parts)
    if content:
        documents.append(
            {
                "id": len(documents) + 1,
                "article": number,
                "title": title,
                "content": content,
                "law_name": selected_law["name"],
                "mst": selected_law["mst"],
                "effective_date": selected_law["effective_date"],
            }
        )

print(f"조문 데이터 {len(documents)}개를 만들었습니다.")
if documents:
    print(documents[0])

조문 데이터 0개를 만들었습니다.


In [ ]:
# 필요한 경우 전체 조문을 확인합니다.
for document in documents:
    print(f"[{document['article']} {document['title']}]")
    print(document["content"])
    print()

## 오류가 발생하면

- `LAW_API_OC가 설정되지 않았습니다`: 프로젝트 루트 `.env`에 `LAW_API_OC`를 추가하고 노트북 커널을 재시작합니다.
- `resultCode`가 `00`이 아님: API 사용 신청 상태, OC 값, 요청 항목을 확인합니다. OC 값은 출력하거나 공유하지 마세요.
- 결과가 0건: 검색어를 짧게 바꾸거나 `PAGE`를 확인합니다.
- 상세 본문이 너무 길면 마지막 셀 대신 `documents[:3]`처럼 일부만 출력합니다.